In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

def run_preprocessing(raw_stock_path, raw_macro_path, output_path):
    print("Starting Preprocessing Pipeline with 4 Macro Indicators...")

    # 1. Load Data
    # Stock data with multi-index columns
    cse = pd.read_csv(raw_stock_path, header=[0, 1], index_col=0)
    # New macro data with Year, Inflation, GDP_Growth, Interest_Rate, Exchange_Rate
    macro = pd.read_csv(raw_macro_path)

    # Flatten CSE Multi-index and fix Date
    cse.columns = cse.columns.get_level_values(0)
    cse.index = pd.to_datetime(cse.index)
    cse = cse.reset_index()

    # 2. Reshape & Clean Macro Data
    # Ensure Year is an integer for merging
    macro['Year'] = macro['Year'].astype(int)
    
    # 3. Feature Engineering
    cse['Year'] = cse['Date'].dt.year

    # 4. Merging
    # Join daily stock data with yearly macro data
    df = pd.merge(cse, macro, on='Year', how='left')

    # 5. Normalization (Requirement 1)
    # List of all 4 features to be scaled
    feature_cols = ['Inflation', 'GDP_Growth', 'Interest_Rate', 'Exchange_Rate']
    
    # Drop rows with missing values in our features or target
    df = df.dropna(subset=feature_cols + ['Close'])

    scaler = MinMaxScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])

    # Save the scaler for the Streamlit front-end
    # Ensure usage of absolute path or correct relative path for models directory
    models_dir = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(output_path))), 'models') if os.path.isabs(output_path) else 'models'
    if not os.path.exists('models') and os.path.exists('../models'):
         models_dir = '../models'

    os.makedirs(models_dir, exist_ok=True)
    with open(os.path.join(models_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    print(f"Scaler saved for app use to {models_dir}")

    # 6. Save Processed Data
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final_df = df[['Date', 'Close'] + feature_cols]
    final_df.to_csv(output_path, index=False)
    
    print(f"Success! Data with 4 features saved to: {output_path}")
    return final_df

if __name__ == "__main__":
    # Adjust paths to be compatible with running from notebooks/ directory
    # If the current working directory is 'notebooks', we need to go up one level
    import os
    if os.path.basename(os.getcwd()) == 'notebooks':
        RAW_STOCK = "../data/raw/cse_historical_data.csv"
        RAW_MACRO = "../data/raw/srilanka_macro_data.csv"
        PROCESSED_OUT = "../data/processed/processed_data.csv"
    else:
        # Assuming run from root
        RAW_STOCK = "data/raw/cse_historical_data.csv"
        RAW_MACRO = "data/raw/srilanka_macro_data.csv"
        PROCESSED_OUT = "data/processed/processed_data.csv"
    
    run_preprocessing(RAW_STOCK, RAW_MACRO, PROCESSED_OUT)

Starting Preprocessing Pipeline with 4 Macro Indicators...
✅ Scaler saved for app use to ../models
✅ Success! Data with 4 features saved to: ../data/processed/processed_data.csv
